# LAB 02 — Experiment 2: MLE vs Laplace smoothing (Bài 16)

- Chia corpus C4 theo **document**: train / valid / test = 8 : 1 : 1 (seed cố định).
- So sánh Bigram và Trigram, mỗi loại với MLE và Laplace, đo perplexity trên train / valid / test.
- Giải thích kết quả nằm ở file `log_probability.md` (mục 16).

In [ ]:
import copy
import gzip
import json
import math
import random
import re
import sys
from collections import Counter

import pandas as pd

SOS = "<s>"
EOS = "</s>"


def split_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    return [s.strip() for s in sentences if s.strip()]


def tokenize(sentence):
    # sys.intern: cac token giong nhau dung chung 1 object -> tiet kiem RAM
    return [sys.intern(t) for t in re.findall(r"[a-z\u00e0-\u1ef9\u0300-\u036f0-9]+", sentence.lower())]


def load_documents(path, text_field="text", limit=None):
    opener = gzip.open if path.endswith(".gz") else open
    docs = []
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            docs.append(json.loads(line)[text_field] if ".json" in path else line)
            if limit is not None and len(docs) >= limit:
                break
    return docs


def prepare_sentences(docs):
    """docs -> list cac cau, moi cau la list token."""
    sentences = []
    for doc in docs:
        for s in split_sentences(doc):
            tokens = tokenize(s)
            if tokens:
                sentences.append(tokens)
    return sentences

In [ ]:
def build_vocabulary(sentences, min_count=1):
    counts = Counter(w for s in sentences for w in s)
    vocab = {w for w, c in counts.items() if c >= min_count}
    vocab.add(EOS)
    return vocab


def pad_sentence(tokens, n):
    return [SOS] * (n - 1) + list(tokens) + [EOS]


def count_ngrams(sentences, n):
    counts = Counter()
    for tokens in sentences:
        padded = pad_sentence(tokens, n)
        for i in range(len(padded) - n + 1):
            counts[tuple(padded[i:i + n])] += 1
    return counts


def count_contexts(ngram_counts):
    contexts = Counter()
    for ngram, c in ngram_counts.items():
        contexts[ngram[:-1]] += c
    return contexts


class NGramLanguageModel:
    def __init__(self, n, smoothing="none"):
        assert n in (1, 2, 3)
        assert smoothing in ("none", "laplace")
        self.n = n
        self.smoothing = smoothing

    def fit(self, sentences):
        self.vocab = build_vocabulary(sentences)
        self.ngram_counts = count_ngrams(sentences, self.n)
        self.context_counts = count_contexts(self.ngram_counts)
        return self

    def with_smoothing(self, smoothing):
        other = copy.copy(self)
        other.smoothing = smoothing
        return other

    def _normalize_context(self, context):
        if self.n == 1:
            return ()
        ctx = list(context)[-(self.n - 1):]
        ctx = [SOS] * (self.n - 1 - len(ctx)) + ctx
        return tuple(ctx)

    def probability(self, context, word):
        ctx = self._normalize_context(context)
        count = self.ngram_counts.get(ctx + (word,), 0)
        ctx_count = self.context_counts.get(ctx, 0)
        if self.smoothing == "laplace":
            return (count + 1) / (ctx_count + len(self.vocab))
        if ctx_count == 0:
            return 0.0
        return count / ctx_count

    def perplexity(self, sentences):
        """Tra ve (perplexity, ty le token co xac suat = 0). PPL = inf neu co token xac suat 0."""
        total_logp, N, zeros = 0.0, 0, 0
        for tokens in sentences:
            padded = pad_sentence(tokens, self.n)
            for i in range(self.n - 1, len(padded)):
                p = self.probability(padded[i - self.n + 1:i], padded[i])
                N += 1
                if p == 0.0:
                    zeros += 1
                else:
                    total_logp += math.log(p)
        ppl = float("inf") if zeros else math.exp(-total_logp / N)
        return ppl, zeros / N

## Load corpus và chia train / valid / test = 8 : 1 : 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CORPUS_PATH = "/content/drive/MyDrive/NLP/lab02/c4-train.00000-of-01024-30K.json.gz"
LIMIT = None   # None = full 30K documents
SEED = 42

Mounted at /content/drive


In [ ]:
docs = load_documents(CORPUS_PATH, limit=LIMIT)

idx = list(range(len(docs)))
random.Random(SEED).shuffle(idx)
n_train = int(0.8 * len(idx))
n_valid = int(0.1 * len(idx))

split_docs = {
    "train": [docs[i] for i in idx[:n_train]],
    "valid": [docs[i] for i in idx[n_train:n_train + n_valid]],
    "test":  [docs[i] for i in idx[n_train + n_valid:]],
}
del docs

data = {name: prepare_sentences(d) for name, d in split_docs.items()}
for name in data:
    n_tok = sum(len(s) for s in data[name])
    print(f"{name:5s}: {len(split_docs[name]):6d} docs, {len(data[name]):8d} sentences, {n_tok:9d} tokens")

train:  24000 docs,   551369 sentences,   8949462 tokens
valid:   3000 docs,    67389 sentences,   1071567 tokens
test :   3000 docs,    67083 sentences,   1068310 tokens


## Huấn luyện: đếm n-gram trên train (MLE và Laplace dùng chung count)

In [ ]:
bigram_mle = NGramLanguageModel(2).fit(data["train"])
trigram_mle = NGramLanguageModel(3).fit(data["train"])

models = {
    "Bigram MLE": bigram_mle,
    "Bigram Laplace": bigram_mle.with_smoothing("laplace"),
    "Trigram MLE": trigram_mle,
    "Trigram Laplace": trigram_mle.with_smoothing("laplace"),
}
V = len(bigram_mle.vocab)
print(f"Vocabulary size (train): V = {V:,}")

Vocabulary size (train): V = 167,124


## Đánh giá perplexity

In [ ]:
rows = []
for name, lm in models.items():
    row = {"Model": name}
    for split in ("train", "valid", "test"):
        ppl, zero_rate = lm.perplexity(data[split])
        row[f"{split.capitalize()} PPL"] = ppl
        row[f"{split.capitalize()} zero-prob %"] = 100 * zero_rate
    rows.append(row)
    print("done:", name)

results = pd.DataFrame(rows)
ppl_cols = ["Model", "Train PPL", "Valid PPL", "Test PPL"]
print()
print(results[ppl_cols].to_string(index=False, float_format=lambda x: f"{x:,.1f}"))

done: Bigram MLE
done: Bigram Laplace
done: Trigram MLE
done: Trigram Laplace

          Model  Train PPL  Valid PPL  Test PPL
     Bigram MLE      161.5        inf       inf
 Bigram Laplace    5,208.5    6,748.7   6,761.9
    Trigram MLE       14.7        inf       inf
Trigram Laplace   28,966.3   47,785.3  48,061.6


In [ ]:
# Ty le token co xac suat = 0 (MLE) va ty le tu ngoai vocabulary (OOV) tren valid/test
zero_cols = ["Model", "Train zero-prob %", "Valid zero-prob %", "Test zero-prob %"]
print(results[zero_cols].to_string(index=False, float_format=lambda x: f"{x:.2f}"))

vocab = bigram_mle.vocab
for split in ("valid", "test"):
    toks = [w for s in data[split] for w in s]
    oov = sum(1 for w in toks if w not in vocab) / len(toks)
    print(f"OOV rate ({split}): {100 * oov:.2f}%")

          Model  Train zero-prob %  Valid zero-prob %  Test zero-prob %
     Bigram MLE               0.00              21.79             21.99
 Bigram Laplace               0.00               0.00              0.00
    Trigram MLE               0.00              57.38             57.42
Trigram Laplace               0.00               0.00              0.00
OOV rate (valid): 1.85%
OOV rate (test): 1.94%


## Lưu kết quả vào `results.csv`

In [ ]:
import csv
import os

RESULTS_PATH = os.path.join(os.path.dirname(CORPUS_PATH), "results.csv")

with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([])  # dong trong ngan cach voi phan Experiment 1
    writer.writerow(["experiment", "model", "train_ppl", "valid_ppl", "test_ppl",
                     "train_zero_pct", "valid_zero_pct", "test_zero_pct"])
    for _, r in results.iterrows():
        writer.writerow(["exp2_smoothing", r["Model"],
                         r["Train PPL"], r["Valid PPL"], r["Test PPL"],
                         round(r["Train zero-prob %"], 4), round(r["Valid zero-prob %"], 4),
                         round(r["Test zero-prob %"], 4)])

print("Da luu:", RESULTS_PATH)

Da luu: /content/drive/MyDrive/NLP/lab02/results.csv
